In [1]:
import pandas as pd
import gseapy as gp
from gseapy.parser import download_library
import numpy as np

# Load in TFs

In [2]:
### Transcription Factors
tfpath = "../resources/allTFs_hg38.txt"

with open(tfpath, 'r') as file:
    tf_list = [line.strip() for line in file]

print(len(tf_list))

1892


# Cell marker genes

In [3]:
def load_pathway(fpath):
    """
    Loads an Enrichr-like database file into a boolean DataFrame.

    Args:
        fpath (str): Path to the Enrichr-like database file.

    Returns:
        pandas.DataFrame: A boolean DataFrame where:
            - Index: Genes
            - Columns: Pathways
            - Values: True if the gene is in the pathway, False otherwise.
    """

    result = []
    with open(fpath,  encoding='utf-8') as f:
        for line in f:
            split_line = [x for x in line.strip().split('\t') if x]  # Remove empty strings directly

            row = {'label': split_line[0]}
            for gene in split_line[1:]:
                row[gene] = 1

            result.append(row)

    df = pd.DataFrame(result)
    df = df.fillna(0.0).set_index('label').astype(bool).T  # Chained operations for clarity
    return df

In [38]:
fpaths = {
    'panglao' : "../resources/PanglaoDB_Augmented_2021.txt",
    'tabula' : "../resources/Tabula_Sapiens.txt",
}

df = []

for key, fpath in fpaths.items():
    tab = load_pathway(fpath)
    tab = tab.astype(int)
    tab = tab.reset_index(names=['gene_name'])

    tab = pd.melt(tab, id_vars='gene_name')

    # drop non-markers
    tab = tab[tab['value'] > 0]
    tab['is_tf'] = tab['gene_name'].isin(tf_list)
    tab = tab.drop(columns='value')

    tab = tab.rename(columns={'label': 'cell_type'})

    tab['cell_type'] = tab['cell_type'].str.lower().str.replace(" ", "_")
    tab['source'] = key

    print(f"{tab.shape=}")
    df.append(tab)

df = pd.concat(df)
print(f"{df.shape=}")
df.head()

tab.shape=(24223, 4)
tab.shape=(46866, 4)
df.shape=(71089, 4)


,gene_name,cell_type,is_tf,source
0,GDF15,acinar_cells,False,panglao
1,RARRES2,acinar_cells,False,panglao
2,TM4SF4,acinar_cells,False,panglao
3,CELA1,acinar_cells,False,panglao
4,GCG,acinar_cells,False,panglao


In [34]:
df[df['cell_type'].str.contains('pluripotent', case=False, na=False)]['cell_type'].unique()

array(['pluripotent_stem_cells'], dtype=object)

In [35]:
cell_type_categories = {
    "fibroblast": [
        "fibroblasts",
        "bladder-fibroblast",
        "large_intestine-fibroblast",
        "lung-fibroblast",
        "pancreas-fibroblast",
        "prostate-fibroblast",
        "salivary_gland-fibroblast",
        "tongue-fibroblast",
        "uterus-fibroblast",
        "vasculature-fibroblast",
        "fat-fibroblast",
        "eye-fibroblast",
        "heart-fibroblast_of_cardiac_tissue",
        "mammary-fibroblast_of_breast"
    ],
    
    "myofibroblast": [
        "myofibroblasts",
        "bladder-myofibroblast_cell",
        "fat-myofibroblast_cell",
        "lung-myofibroblast_cell"
    ],
    
    "skeletal_muscle": [
        "muscle-skeletal_muscle_satellite_stem_cell",
        "skin-cell_of_skeletal_muscle",
    ],
    
    "myocyte": [
        "cardiomyocytes",
        "myocytes",
    ],
    
    "myoblast": [
        "myoblasts",
    ],
    
    "muscle": [
       'airway_smooth_muscle_cells',
       'pulmonary_vascular_smooth_muscle_cells',
       'smooth_muscle_cells',
       'vascular_smooth_muscle_cells',
       'bladder-smooth_muscle_cell',
       'lung-bronchial_smooth_muscle_cell',
       'lung-vascular_associated_smooth_muscle_cell',
       'muscle-capillary_endothelial_cell',
       'muscle-cd4-positive,_alpha-beta_t_cell',
       'muscle-cd8-positive,_alpha-beta_t_cell',
       'muscle-endothelial_cell_of_artery',
       'muscle-endothelial_cell_of_lymphatic_vessel',
       'muscle-endothelial_cell_of_vascular_tree', 
        'muscle-erythrocyte',
       'muscle-fast_muscle_cell', 
        'muscle-macrophage', 
        'muscle-mast_cell',
       'muscle-mature_nk_t_cell', 
        'muscle-mesenchymal_stem_cell',
       'muscle-pericyte_cell',
       'muscle-slow_muscle_cell', 
        'muscle-t_cell', 
        'muscle-tendon_cell',
       'thymus-fast_muscle_cell',
       'thymus-vascular_associated_smooth_muscle_cell',
       'vasculature-smooth_muscle_cell',
       'mammary-vascular_associated_smooth_muscle_cell',
       'uterus-vascular_associated_smooth_muscle_cell',
       'trachea-smooth_muscle_cell', 
        'tongue-tongue_muscle_cell',
       'prostate-smooth_muscle_cell', 
        'fat-smooth_muscle_cell',
       'skin-muscle_cell', 
        'heart-cardiac_muscle_cell',
       'heart-smooth_muscle_cell', 
        'lung-smooth_muscle_cell',
       'muscle-mesothelial_cell', 
        'muscle-smooth_muscle_cell', 
        'skin-smooth_muscle_cell'
    ],
    
    "ipsc": [
        "pluripotent_stem_cells",
    ]
}

# Reverse mapping: each cell type → its category
cell_type_to_category = {
    cell_type: category
    for category, cell_types in cell_type_categories.items()
    for cell_type in cell_types
}

df['category'] = df['cell_type'].map(cell_type_to_category)
df = df[df['category'].notna()]

print(df['category'].value_counts().to_string())

category
muscle             3964
fibroblast         1532
myofibroblast       401
myocyte             337
skeletal_muscle     200
myoblast            126
ipsc                112


In [36]:
outpath = "../resources/clean_marker_genes.csv"
df = df.reset_index(drop=True)
df.to_csv(outpath, index=False)
df.head()

,gene_name,cell_type,is_tf,source,category
0,LUM,airway_smooth_muscle_cells,False,panglao,muscle
1,COL1A1,airway_smooth_muscle_cells,False,panglao,muscle
2,COL5A2,airway_smooth_muscle_cells,False,panglao,muscle
3,COL5A1,airway_smooth_muscle_cells,False,panglao,muscle
4,FNDC1,airway_smooth_muscle_cells,False,panglao,muscle


In [37]:
df['is_tf'].value_counts()

is_tf
False    6234
True      438
Name: count, dtype: int64

# Other gene sets for scoring

### GO Biological Processes

In [4]:
### GO biological processes
path_2023 = "../../resources/GO_Biological_Process_2023.txt"
path_2025 = "/home/jrcwycy/.cache/gseapy/Enrichr.GO_Biological_Process_2025.gmt"

geneset_dict = {}

with open(path_2025, "r") as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 1:
            term = parts[0].strip()
            genes = parts[2:] if len(parts) > 2 else []
            genes = [g.strip() for g in genes if g.strip()]
            geneset_dict[term] = genes

geneset = pd.DataFrame.from_dict(geneset_dict, orient='index').T

In [5]:
[col for col in geneset if 'hippo' in col.lower()]

['Hippo Signaling (GO:0035329)',
 'Negative Regulation of Hippo Signaling (GO:0035331)',
 'Positive Regulation of Hippo Signaling (GO:0035332)',
 'Regulation of Hippo Signaling (GO:0035330)']

In [5]:
### In GO 2025
pos_prolif = geneset[[
    'Positive Regulation of Fibroblast Proliferation (GO:0048146)',
    'Positive Regulation of Cell Population Proliferation (GO:0008284)',
]]

neg_prolif = geneset[[
    'Negative Regulation of Fibroblast Proliferation (GO:0048147)',
    'Negative Regulation of Cell Population Proliferation (GO:0008285)',
]]

# G1/S
g1s_transition = geneset[[
    'G1/S Transition of Mitotic Cell Cycle (GO:0000082)',
    'Cell Cycle G1/S Phase Transition (GO:0044843)',
    'Regulation of G1/S Transition of Mitotic Cell Cycle (GO:2000045)',
    'Regulation of Cell Cycle G1/S Phase Transition (GO:1902806)',
]]

checkpoint_signaling = geneset[[
    'Mitotic G1 DNA Damage Checkpoint Signaling (GO:0031571)',
    'Mitotic G1/S Transition Checkpoint Signaling (GO:0044819)',
    'Mitotic G2 DNA Damage Checkpoint Signaling (GO:0007095)',
    'Mitotic G2/M Transition Checkpoint (GO:0044818)',
    'Mitotic Cell Cycle Checkpoint Signaling (GO:0007093)',
    'Regulation of Cell Cycle Checkpoint (GO:1901976)',
]]

g2m_transition = geneset[[
    'G2/M Transition of Mitotic Cell Cycle (GO:0000086)',
    'Cell Cycle G2/M Phase Transition (GO:0044839)',
    'Regulation of G2/M Transition of Mitotic Cell Cycle (GO:0010389)',
]]

general_cc = geneset[[
    'Regulation of Cell Cycle (GO:0051726)',
    'Regulation of Cell Cycle Process (GO:0010564)',
]]

hippo = geneset[[
    'Hippo Signaling (GO:0035329)',
    'Regulation of Hippo Signaling (GO:0035330)',
]]

In [8]:
# read in quiescence genes
fpath = "../resources/genes_for_scoring.csv"
df = pd.read_csv(fpath, index_col=0)

print(df.columns.tolist())

qdf = df[['Circadian', 'Core_reprogramming', 'Quiescence_TFs', 'AS', 'Chromatin_remodeling']]
qdf.head()

['TFs', 'Fibroblasts', 'Myofibroblasts', 'Myogenic', 'cell_cycle', 'AS', 'Circadian', 'Core_reprogramming', 'Senescence_GO', 'Senescence_UP', 'Senescence_DOWN', 'DNA_damage', 'Chromatin_remodeling', 'TF_activity', 'Pos_apoptosis', 'Neg_apoptosis', 'Quiescence_TFs']


,Circadian,Core_reprogramming,Quiescence_TFs,AS,Chromatin_remodeling
0,TEF,HDAC9,ATF3,CWC27,H2AJ
1,SFPQ,POLR2C,ATF4,DDX20,HCFC1
2,THRAP3,PTN,CTNNB1,RBM3,INO80E
3,NKX2-1,JUN,BACH2,RBM5,HDAC8
4,LOC102724428,MED31,JUN,RBM23,BAZ2A


### Reactome

In [2]:
database = "Reactome_Pathways_2024"

library = download_library(database)

library = pd.DataFrame.from_dict(library, orient='index').T
library.head()

,2-LTR Circle Formation,A Tetrasaccharide Linker Sequence Is Required for GAG Synthesis,ABC Transporter Disorders,ABC Transporters in Lipid Homeostasis,ABC-family Proteins Mediated Transport,ADORA2B Mediated Anti-Inflammatory Cytokines Production,ADP Signalling Through P2Y Purinoceptor 1,ADP Signalling Through P2Y Purinoceptor 12,AKT Phosphorylates Targets in the Cytosol,AKT Phosphorylates Targets in the Nucleus,...,tRNA Aminoacylation,tRNA Modification in the Mitochondrion,tRNA Modification in the Nucleus and Cytosol,tRNA Processing,tRNA Processing in the Mitochondrion,tRNA Processing in the Nucleus,"tRNA-derived Small RNA (tsRNA or tRNA-related Fragment, tRF) Biogenesis",trans-Golgi Network Vesicle Budding,vRNA Synthesis,vRNP Assembly
0,REV,B3GAT3,ABCG8,ABCA2,ABCC4,GNG13,PLA2G4A,P2RY12,CDKN1B,CREB1,...,MARS1,TRIT1,TRMT10A,PUS3,TRMT10C,SEC13,TRP-AGG1-1,SNAPIN,PA,PA
1,XRCC5,B3GAT2,ABCC2,ABCG8,ABCG8,GNG12,GNG13,GNG13,GSK3B,RPS6KB2,...,HARS2,TRMT10C,NSUN2,CPSF1,ELAC2,CPSF1,TRR-CCG1-1,AP1S1,NP,NP
2,XRCC6,B3GAT1,KCNJ11,ABCA3,ABCC5,CREB1,GNG12,GNAI1,GSK3A,AKT3,...,RARS1,TRMU,PUS3,TYW5,TRNT1,CPSF4,TRA-TGC1-1,HSPA8,NS,HSP90AA1
3,XRCC4,BGN,ABCC8,ABCD3,ABCC2,IL6,MAPK14,GNG12,BAD,AKT1,...,HARS1,PRORP,TRIT1,CPSF4,MT-ATP8,ELAC2,TRV-AAC1-1,YIPF6,PARP1,NS
4,PSIP1,B3GALT6,ABCC9,PEX19,PEX19,ADORA2B,GNB1,GNAI2,CDKN1A,AKT2,...,RARS2,PUS1,METTL1,NUP214,MT-TA,NDC1,ELAC2,AP1B1,PB2,IPO5


In [4]:
# library.to_csv("../resources/reactome_pathways_2024.csv")

In [10]:
notch_signaling = ['Signaling by NOTCH', 'Signaling by NOTCH1', 'Signaling by NOTCH2', 'Signaling by NOTCH3', 'Signaling by NOTCH4']

notch_df = library[notch_signaling]
notch_df.head()

foxo_signaling = library[[col for col in library if "foxo" in col.lower()]]

p53_signaling = library[['Regulation of TP53 Activity', 'TP53 Regulates Transcription of Genes Involved in G1 Cell Cycle Arrest']]

dream = library[['Transcription of E2F Targets Under Negative Control by DREAM Complex', 
                 'Transcription of E2F Targets Under Neg Control by P107 (RBL1) and P130 (RBL2) in Complex With HDAC1']]

senescence_stress = library['DNA Damage Telomere Stress Induced Senescence'].unique()
sasp = library['Senescence-Associated Secretory Phenotype (SASP)'].unique()

In [13]:
categories = {
    "G1/S_transition": list(set(g1s_transition.stack().unique().tolist())),
    "G2/M_transition": list(set(g2m_transition.stack().unique().tolist())),
    "CC_checkpoints": list(set(checkpoint_signaling.stack().unique().tolist())),
    "Cell_cycle_reg": list(set(general_cc.stack().unique().tolist())),
    "Pos_proliferation": list(set(pos_prolif.stack().unique().tolist())),
    "Neg_proliferation": list(set(neg_prolif.stack().unique().tolist())),
    "quiescence": qdf['Quiescence_TFs'].unique(),
    "AS": qdf['AS'].unique(),
    'Chromatin': qdf['Chromatin_remodeling'].unique(),
    'Circadian': qdf['Circadian'].unique(),
    'Core_repro': qdf['Core_reprogramming'].unique(),
    "NOTCH_signaling": list(set(notch_df.stack().unique().tolist())),
    'Asymm_PCP': library['Asymmetric Localization of PCP Proteins'].unique(),
    "FOXO": list(set(foxo_signaling.stack().unique().tolist())),
    "p53_acyivity": list(set(p53_signaling.stack().unique().tolist())),
    "DREAM/negE2F": list(set(dream.stack().unique().tolist())),
    "senes_stress": senescence_stress,
    'sasp': sasp,
}

for key, gene_list in categories.items():
    print(f"N genes for {key}: {len(gene_list)}")
    
# make into a dataframe
cleaned_categories = {k: list(np.ravel(v)) for k, v in categories.items()}
max_len = max(len(v) for v in cleaned_categories.values())
padded = {k: v + [None] * (max_len - len(v)) for k, v in cleaned_categories.items()}

df = pd.DataFrame(padded)
df.head()

N genes for G1/S_transition: 196
N genes for G2/M_transition: 72
N genes for CC_checkpoints: 71
N genes for Cell_cycle_reg: 431
N genes for Pos_proliferation: 484
N genes for Neg_proliferation: 379
N genes for quiescence: 60
N genes for AS: 282
N genes for Chromatin: 512
N genes for Circadian: 61
N genes for Core_repro: 145
N genes for NOTCH_signaling: 209
N genes for Asymm_PCP: 52
N genes for FOXO: 67
N genes for p53_acyivity: 169
N genes for DREAM/negE2F: 22
N genes for senes_stress: 61
N genes for sasp: 82


,G1/S_transition,G2/M_transition,CC_checkpoints,Cell_cycle_reg,Pos_proliferation,Neg_proliferation,quiescence,AS,Chromatin,Circadian,Core_repro,NOTCH_signaling,Asymm_PCP,FOXO,p53_acyivity,DREAM/negE2F,senes_stress,sasp
0,FBXW7,CDC25A,CHFR,YTHDF2,EMP2,IGFBP5,ATF3,CWC27,H2AJ,TEF,HDAC9,FBXW7,SMURF2,TXN,MAPK11,CDC25A,H2AZ2,H2AZ2
1,CCNP,CDK6,CDK1,PRKCA,TNXB,ZBTB7C,ATF4,DDX20,HCFC1,SFPQ,POLR2C,ST3GAL4,SMURF1,PLXNA4,TPX2,CDK1,ACD,H2BC9
2,EZH2,NES,MRNIP,OBSL1,CX3CL1,NTRK1,CTNNB1,RBM3,INO80E,THRAP3,PTN,FCER2,WNT5A,SOD2,TAF9B,E2F1,H2BC9,JUN
3,APBB1,CDK1,DGKZ,LOC102724428,CCND2,KDF1,BACH2,RBM5,HDAC8,NKX2-1,JUN,PSMA7,SCRIB,KAT2B,DYRK2,LIN52,H2BC4,UBE2C
4,DDX3X,FBXL15,BARD1,MAGEA4,LIG4,BTG3,JUN,RBM23,BAZ2A,LOC102724428,MED31,TBL1X,PSMC6,IGFBP1,MDM4,RBL1,H2BC5,H2BC4


In [14]:
outpath = "../resources/genes_for_scoring.csv"
df = df.reset_index(drop=True)
df.to_csv(outpath, index=False)
df.head()

,G1/S_transition,G2/M_transition,CC_checkpoints,Cell_cycle_reg,Pos_proliferation,Neg_proliferation,quiescence,AS,Chromatin,Circadian,Core_repro,NOTCH_signaling,Asymm_PCP,FOXO,p53_acyivity,DREAM/negE2F,senes_stress,sasp
0,FBXW7,CDC25A,CHFR,YTHDF2,EMP2,IGFBP5,ATF3,CWC27,H2AJ,TEF,HDAC9,FBXW7,SMURF2,TXN,MAPK11,CDC25A,H2AZ2,H2AZ2
1,CCNP,CDK6,CDK1,PRKCA,TNXB,ZBTB7C,ATF4,DDX20,HCFC1,SFPQ,POLR2C,ST3GAL4,SMURF1,PLXNA4,TPX2,CDK1,ACD,H2BC9
2,EZH2,NES,MRNIP,OBSL1,CX3CL1,NTRK1,CTNNB1,RBM3,INO80E,THRAP3,PTN,FCER2,WNT5A,SOD2,TAF9B,E2F1,H2BC9,JUN
3,APBB1,CDK1,DGKZ,LOC102724428,CCND2,KDF1,BACH2,RBM5,HDAC8,NKX2-1,JUN,PSMA7,SCRIB,KAT2B,DYRK2,LIN52,H2BC4,UBE2C
4,DDX3X,FBXL15,BARD1,MAGEA4,LIG4,BTG3,JUN,RBM23,BAZ2A,LOC102724428,MED31,TBL1X,PSMC6,IGFBP1,MDM4,RBL1,H2BC5,H2BC4
